In [8]:
from pathlib import Path
import re
import sys
import traceback
import pandas as pd

# Optional progress bar
try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x  # no-op if tqdm not installed

# ---- PyMuPDF (primary) ----
try:
    import fitz  # PyMuPDF
    HAVE_FITZ = True
    # Try to silence noisy MuPDF console warnings (version-dependent)
    try:
        fitz.TOOLS.mupdf_display_errors(False)
    except Exception:
        try:
            fitz.TOOLS.mupdf_warnings(False)
        except Exception:
            pass
except Exception:
    HAVE_FITZ = False

# ---- pdfminer.six (fallback) ----
try:
    from pdfminer.high_level import extract_text as pdfminer_extract_text
    HAVE_PDFMINER = True
except Exception:
    HAVE_PDFMINER = False


# ---------------- Utilities ----------------
# Build a compiled regex to strip characters that break XML/Excel:
# 1) C0 and C1 control ranges, incl. NULL
# 2) Non-characters like U+FDD0..U+FDEF, and U+FFFE/U+FFFF (and their plane variants)
CTRL_XML_PATTERN = re.compile(
    "["                                 # begin char class
    "\u0000-\u001F"                     # C0 controls (incl. NULL)
    "\u007F-\u009F"                     # C1 controls
    "\uFDD0-\uFDEF"                     # non-characters
    "\uFFFE\uFFFF"                      # non-characters
    "]"
)

# Add plane-specific non-characters (last two code points on each plane)
# We'll strip them on the fly with a function (cheaper to check explicitly than regex ranges across planes)


def _strip_plane_nonchars(s: str) -> str:
    # Remove U+nFFFE and U+nFFFF for n in 0..16 (BMP..Plane 16)
    if not s:
        return s
    # Precompute once
    if not hasattr(_strip_plane_nonchars, "_bad"):
        bad = []
        for n in range(0x00, 0x11):  # planes 0..16
            bad.append(chr((n << 16) + 0xFFFE))
            bad.append(chr((n << 16) + 0xFFFF))
        _strip_plane_nonchars._bad = set(bad)
    return "".join(ch for ch in s if ch not in _strip_plane_nonchars._bad)


def excel_xml_safe_text(s: str) -> str:
    """Make text safe for writing to .xlsx (no NULL/control/non-characters)."""
    if s is None:
        return ""
    # Replace NBSP and common oddities
    s = s.replace("\xa0", " ")
    # Remove XML-breaking controls
    s = CTRL_XML_PATTERN.sub(" ", s)
    # Remove plane non-characters
    s = _strip_plane_nonchars(s)
    # Collapse runs of whitespace
    s = re.sub(r"[ \t\f\r\v]+", " ", s)
    # Keep paragraph breaks but de-duplicate them
    s = re.sub(r"\n{2,}", "\n", s)
    return s.strip()


def get_first_page_text_pymupdf(pdf_path: Path) -> str:
    if not HAVE_FITZ:
        raise RuntimeError("PyMuPDF (fitz) not available")
    doc = fitz.open(pdf_path)
    try:
        if doc.needs_pass:
            raise PermissionError("PDF is encrypted and requires a password")
        if doc.page_count == 0:
            return ""
        page = doc.load_page(0)
        text = page.get_text("text")
        return excel_xml_safe_text(text)
    finally:
        doc.close()


def get_first_page_text_pdfminer(pdf_path: Path) -> str:
    if not HAVE_PDFMINER:
        raise RuntimeError("pdfminer.six not available")
    text = pdfminer_extract_text(str(pdf_path), page_numbers=[0])
    return excel_xml_safe_text(text)


def extract_first_pages(
    input_dir="test1",
    recursive=True,
    out_dir="output",
    csv_name="pdf_firstpages.csv",
    xlsx_name="pdf_firstpages.xlsx",
    error_log_txt="pdf_errors.log",
    error_log_csv="pdf_errors.csv"
):
    base_dir = Path.cwd()
    input_dir = Path(input_dir) if Path(input_dir).is_absolute() else base_dir / input_dir
    out_dir = Path(out_dir) if Path(out_dir).is_absolute() else base_dir / out_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input folder not found: {input_dir}")

    pdf_iter = input_dir.rglob("*.pdf") if recursive else input_dir.glob("*.pdf")
    pdf_paths = sorted([p for p in pdf_iter if p.is_file()])

    print(f"Found {len(pdf_paths)} PDF files under '{input_dir}'. Starting extraction...")

    rows, errors = [], []
    for pdf_path in tqdm(pdf_paths, desc="Processing PDFs"):
        rel_path = pdf_path.relative_to(base_dir)
        try:
            text = None
            # Try PyMuPDF first
            if HAVE_FITZ:
                try:
                    text = get_first_page_text_pymupdf(pdf_path)
                except Exception:
                    text = None  # fall back

            # Fall back to pdfminer if needed
            if text is None and HAVE_PDFMINER:
                text = get_first_page_text_pdfminer(pdf_path)

            if text is None:
                raise RuntimeError("Failed to extract with both PyMuPDF and pdfminer.six")

            rows.append({
                "file_name": pdf_path.name,
                "relative_path": str(rel_path),
                "first_page_text": text
            })

        except Exception as e:
            tb = traceback.format_exc()
            errors.append({
                "file_name": pdf_path.name,
                "relative_path": str(rel_path),
                "error_type": type(e).__name__,
                "error_message": str(e)
            })
            # Append to text log incrementally
            with (out_dir / error_log_txt).open("a", encoding="utf-8") as f:
                f.write(f"=== {rel_path} ===\n{tb}\n")

    # Save main table (CSV + XLSX)
    df = pd.DataFrame(rows, columns=["file_name", "relative_path", "first_page_text"])

    csv_path = out_dir / csv_name
    xlsx_path = out_dir / xlsx_name
    # CSV: use utf-8-sig to make Excel happier on Windows, safe on macOS too
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")

    # XLSX: openpyxl backend
    # (After our sanitization, this should no longer raise the XML control-char error)
    df.to_excel(xlsx_path, index=False, engine="openpyxl")

    # Errors (if any)
    if errors:
        df_err = pd.DataFrame(errors, columns=["file_name", "relative_path", "error_type", "error_message"])
        df_err.to_csv(out_dir / error_log_csv, index=False, encoding="utf-8-sig")
        print(f"\nCompleted with {len(errors)} errors. See:")
        print(f" - {out_dir / error_log_txt}")
        print(f" - {out_dir / error_log_csv}")
    else:
        print("\nCompleted with no errors.")

    print(f"Output saved to:\n - {csv_path}\n - {xlsx_path}")
    return df


# ---------- Run it (notebook-friendly) ----------
df_result = extract_first_pages(
    input_dir="test1",     # or absolute path
    recursive=True,
    out_dir="output"
)
df_result.head()


Found 420 PDF files under '/Users/pz/VScode/OSL_source_ana/test1'. Starting extraction...


Processing PDFs: 100%|██████████| 420/420 [00:05<00:00, 73.10it/s]



Completed with no errors.
Output saved to:
 - /Users/pz/VScode/OSL_source_ana/output/pdf_firstpages.csv
 - /Users/pz/VScode/OSL_source_ana/output/pdf_firstpages.xlsx


,file_name,relative_path,first_page_text
0,& Liu-1994-An outline of physical geography in...,test1/& Liu-1994-An outline of physical geogra...,GeoJournal 34.1 7-30 © 1994 (Sep) by Kluwer Ac...
1,(Routledge Physical Environment Series) Nichol...,test1/(Routledge Physical Environment Series) ...,
2,0218-中国第十九届释光与电子自旋共振测年学术讨论会（第二号通知）.pdf,test1/0218-中国第十九届释光与电子自旋共振测年学术讨论会（第二号通知）.pdf,中国第十九届释光与电子自旋共振测年学术讨论会 （第二号通知） ———————————————...
3,0331沈勤径V5-博后基金.pdf,test1/0331沈勤径V5-博后基金.pdf,项目名称 澜沧江昌都段河流阶地年代学及其发育机制研究 1．选题依据（国内外研究现状及选题价值...
4,1-s2.0-S0012821X15007062-main.pdf,test1/1-s2.0-S0012821X15007062-main.pdf,Earth and Planetary Science Letters 433 (2016)...


In [9]:
"""
Send first-page text (from output/pdf_firstpages.*) to DeepSeek to:
  1) Decide if it's a research paper (vs CV, slides, etc.). Only create a new name if research paper.
  2) Extract authors, year, journal, title; construct a new filename like:
       Bauska et al.-2019-Nature-PAPERTITLE.pdf
     Journal rule: If single word (e.g., Nature), use it as-is.
                   If multi-word (e.g., Quaternary Science Reviews), use initials (e.g., QSR).
     Authors rule for filename:
       - 1 author: "Surname Initials"
       - 2 authors: "Surname Initials & Surname Initials"
       - ≥3 authors: "Surname Initials et al."
  3) Iterate over all rows and build a mapping table (old_name → suggested_new_name).
  4) Uses your DeepSeek API key (below) for a chat completion.
  5) Does NOT rename files on disk; only writes mapping tables.

Requirements:
  pip install pandas openpyxl requests tqdm

Notes:
  - This script accepts either CSV or XLSX produced earlier (pdf_firstpages.csv/xlsx).
  - It writes:
      output/deepseek_filename_suggestions.csv
      output/deepseek_filename_suggestions.xlsx
  - Robust filename sanitizer is applied on model output.
"""

import os
import json
import time
from pathlib import Path
from typing import Optional, Dict, Any

import pandas as pd
import requests
from tqdm import tqdm

# =========================
# Configuration
# =========================
INPUT_XLSX = Path("output/pdf_firstpages.xlsx")
INPUT_CSV  = Path("output/pdf_firstpages.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV  = OUTPUT_DIR / "deepseek_filename_suggestions.csv"
OUT_XLSX = OUTPUT_DIR / "deepseek_filename_suggestions.xlsx"
RAW_JSONL = OUTPUT_DIR / "deepseek_raw_responses.jsonl"   # optional audit trail

# DeepSeek API (OpenAI-compatible chat endpoint)
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"

# Use the key you provided; you may switch to an env var if you prefer.
DEEPSEEK_API_KEY = "sk-5d269924fa9e48888d7109f5971b5f1e"

# Model name; adjust if your account uses a different identifier (e.g., "deepseek-chat", "deepseek-coder")
DEEPSEEK_MODEL = "deepseek-chat"

# Rate limiting / retry
MAX_RETRIES = 3
RETRY_BACKOFF = 2.0  # seconds


# =========================
# Helpers
# =========================
def load_input_table() -> pd.DataFrame:
    if INPUT_XLSX.exists():
        df = pd.read_excel(INPUT_XLSX)
    elif INPUT_CSV.exists():
        df = pd.read_csv(INPUT_CSV)
    else:
        raise FileNotFoundError("Cannot find output/pdf_firstpages.xlsx or output/pdf_firstpages.csv")
    # Expecting columns: file_name, relative_path, first_page_text
    needed = {"file_name", "relative_path", "first_page_text"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Input table missing required columns: {missing}")
    return df


def sanitize_filename_component(s: str) -> str:
    """
    Make a string safe for filenames:
      - replace forbidden chars:  / \ : * ? " < > | and control chars
      - collapse whitespace
      - trim
      - limit length
    """
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    # Replace forbidden filename characters with spaces
    forbidden = r'\/:*?"<>|'
    t = "".join((" " if ch in forbidden else ch) for ch in s)
    # Remove control chars
    t = "".join(ch for ch in t if ch >= " " and ch != "\x7f")
    # Collapse whitespace
    t = " ".join(t.split())
    # Optional: replace spaces with underscores (comment out if you prefer spaces)
    # t = t.replace(" ", "_")
    # Trim length (most filesystems handle up to 255; keep a safe budget)
    if len(t) > 160:
        t = t[:160].rstrip()
    return t


def build_prompt(file_name: str, first_page_text: str) -> str:
    """
    Instruction-prompt for DeepSeek to (1) classify, (2) extract, (3) construct filename.
    The model must return STRICT JSON (no extra text).
    """
    return f"""
You are given the FIRST PAGE raw text of a PDF and its original file name.
Your tasks:
1) Decide if it is a peer-reviewed research paper article (journal article). If it looks like a CV, thesis title page, book chapter cover, conference presentation slides, poster, manual, report, or anything else, mark it as not a research paper.
2) If it IS a research paper, extract:
   - authors: a list of author names in order as they appear (e.g., ["Matthew N. Bauska", "Thomas Bauska", ...]).
   - year: the publication year as a 4-digit integer.
   - journal_full: the journal's full name (e.g., "Quaternary Science Reviews", "Nature", "Science", "Proceedings of the National Academy of Sciences").
   - title: the article title as shown on the first page (remove leading/trailing newlines/spaces).
3) Construct:
   - journal_short: If journal_full has exactly one word (e.g., "Nature"), use that word as-is. If it has multiple words (e.g., "Quaternary Science Reviews"), create an uppercase abbreviation using the first letter of each word, ignoring short stopwords like "of", "the", "and", "&", "in", "for". Examples: "Quaternary Science Reviews" -> "QSR", "Proceedings of the National Academy of Sciences" -> "PNAS".
   - authors_for_filename: Apply the rules:
       • If 1 author, use "Surname Initials" (e.g., "Bauska MN").
       • If 2 authors, use "Surname Initials & Surname Initials" (e.g., "Bauska MN & Osman MB").
       • If 3 or more authors, use "Surname Initials et al." (e.g., "Bauska MN et al.").
     Use the *family name / surname* and initials of given names. Preserve diacritics in names where present.
   - title_for_filename: A concise version of the title with punctuation removed that would be invalid in filenames. Keep the main words; don't exceed ~120 characters.
   - new_filename: "{'{authors_for_filename}'}-{ '{year}' }-{ '{journal_short}' }-{ '{title_for_filename}' }"
     Do NOT include the .pdf extension; we will add it.

Rules:
- Return STRICT JSON only, with keys below; no commentary.
- If NOT a research paper, set "is_research_paper": false and leave other fields empty strings or empty arrays.
- If unsure about year/journal/title, make a best attempt from the first page; do NOT invent beyond the provided text.
- Preserve Unicode (Chinese authors/titles ok). Do NOT truncate surnames to pinyin; keep as-is.

JSON schema to return:
{{
  "is_research_paper": true/false,
  "authors": ["..."],
  "year": "YYYY",
  "journal_full": "...",
  "journal_short": "...",
  "title": "...",
  "authors_for_filename": "...",
  "title_for_filename": "...",
  "new_filename": "..."
}}

Context:
- original_file_name: "{sanitize_filename_component(file_name)}"
- first_page_text:
\"\"\"
{first_page_text[:8000]}
\"\"\"  # first page text (truncated if very long)
""".strip()


def call_deepseek(prompt: str) -> Dict[str, Any]:
    """
    Calls DeepSeek chat completion with the given prompt.
    Returns parsed JSON dict as specified by the prompt (or raises on failure).
    """
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "system", "content": "You are a meticulous bibliographic extraction assistant. Return ONLY valid JSON."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.0,
        "response_format": {"type": "json_object"}  # if supported; otherwise the prompt enforces JSON
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(DEEPSEEK_API_URL, headers=headers, data=json.dumps(payload), timeout=60)
            if resp.status_code == 200:
                data = resp.json()
                # OpenAI-compatible shape:
                content = data["choices"][0]["message"]["content"]
                # Optionally append raw response to an audit log
                with open(RAW_JSONL, "a", encoding="utf-8") as f:
                    f.write(json.dumps({"prompt_hash": hash(prompt), "response": data}, ensure_ascii=False) + "\n")
                # Parse JSON content
                return json.loads(content)
            else:
                last_err = RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
        except Exception as e:
            last_err = e
        # backoff
        time.sleep(RETRY_BACKOFF * attempt)

    raise last_err if last_err else RuntimeError("Unknown DeepSeek error")


def assemble_suggested_filename(record: Dict[str, Any]) -> str:
    """
    Defensive post-processing of model output to ensure filesystem-safe name.
    Adds .pdf extension. If not research paper or missing parts, returns "".
    """
    if not record or not record.get("is_research_paper"):
        return ""
    parts = [
        sanitize_filename_component(record.get("authors_for_filename", "")),
        sanitize_filename_component(record.get("year", "")),
        sanitize_filename_component(record.get("journal_short", "")),
        sanitize_filename_component(record.get("title_for_filename", "")),
    ]
    # Remove empties and join with '-'
    base = "-".join([p for p in parts if p])
    base = base.strip("- ")
    if not base:
        return ""
    # Ensure extension
    return base + ".pdf"


# =========================
# Main execution
# =========================
df = load_input_table()

rows_out = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="DeepSeek extraction"):
    orig_name = str(row.get("file_name", ""))
    rel_path  = str(row.get("relative_path", ""))
    first_txt = str(row.get("first_page_text", "")) if pd.notna(row.get("first_page_text", "")) else ""

    prompt = build_prompt(orig_name, first_txt)

    try:
        result = call_deepseek(prompt)
    except Exception as e:
        # On failure, record an error row
        rows_out.append({
            "file_name": orig_name,
            "relative_path": rel_path,
            "is_research_paper": False,
            "authors": "",
            "year": "",
            "journal_full": "",
            "journal_short": "",
            "title": "",
            "authors_for_filename": "",
            "title_for_filename": "",
            "suggested_new_filename": "",
            "api_error": str(e)
        })
        continue

    # Defensive normalization
    is_paper = bool(result.get("is_research_paper", False))
    # Compose filename locally as a safeguard (even if model returned new_filename)
    suggested = assemble_suggested_filename(result)

    rows_out.append({
        "file_name": orig_name,
        "relative_path": rel_path,
        "is_research_paper": is_paper,
        "authors": "; ".join(result.get("authors", [])) if isinstance(result.get("authors"), list) else str(result.get("authors", "")),
        "year": str(result.get("year", "")),
        "journal_full": result.get("journal_full", ""),
        "journal_short": result.get("journal_short", ""),
        "title": result.get("title", ""),
        "authors_for_filename": result.get("authors_for_filename", ""),
        "title_for_filename": result.get("title_for_filename", ""),
        "suggested_new_filename": suggested or sanitize_filename_component(result.get("new_filename", "")) + ("" if str(result.get("new_filename", "")).endswith(".pdf") else ".pdf"),
        "api_error": ""
    })
    # print the first few for inspection
    if len(rows_out) <= 5:
        print(json.dumps(rows_out[-1], ensure_ascii=False, indent=2))

# Save outputs
df_out = pd.DataFrame(rows_out, columns=[
    "file_name", "relative_path", "is_research_paper",
    "authors", "year", "journal_full", "journal_short", "title",
    "authors_for_filename", "title_for_filename", "suggested_new_filename",
    "api_error"
])

df_out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
df_out.to_excel(OUT_XLSX, index=False, engine="openpyxl")
print(f"Saved:\n  {OUT_CSV}\n  {OUT_XLSX}")

# At this stage, you can inspect 'suggested_new_filename' and decide whether to actually rename files.
# (Intentionally not renaming files per your instruction.)


DeepSeek extraction: 100%|██████████| 420/420 [1:12:07<00:00, 10.30s/it] 

Saved:
  output/deepseek_filename_suggestions.csv
  output/deepseek_filename_suggestions.xlsx


# 重命名文件

In [10]:
"""
Rename PDF files according to 'deepseek_filename_suggestions' table.

Rules:
- Only rename rows where `is_research_paper` is True.
- Use `relative_path` to locate the original file.
- Use `suggested_new_filename` as the target name (in the same folder).
- Avoid overwriting: if the target exists, append a de-duplicating suffix.
- Logs every action to: output/rename_results.csv

Requirements:
  pip install pandas openpyxl
"""

from pathlib import Path
import pandas as pd
import shutil
import re
import time

# ------------------ Config ------------------
# Table produced by the previous step
SUGGESTIONS_XLSX = Path("output/deepseek_filename_suggestions.xlsx")
SUGGESTIONS_CSV  = Path("output/deepseek_filename_suggestions.csv")

# Base directory used to resolve 'relative_path'
BASE_DIR = Path.cwd()

# Where to write the rename report
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_CSV = OUTPUT_DIR / "rename_results.csv"

# Safety valve: set to True to preview without renaming
DRY_RUN = False

# ---------------- Utilities -----------------
FORBIDDEN = r'\/:*?"<>|'

def sanitize_basename(name: str) -> str:
    """Ensure the suggested name is a safe plain filename (no directories)."""
    if not isinstance(name, str):
        name = "" if name is None else str(name)
    # Strip any path components (keep only the last component)
    name = Path(name).name
    # Replace forbidden chars
    name = "".join((" " if ch in FORBIDDEN else ch) for ch in name)
    # Remove control chars
    name = "".join(ch for ch in name if ch >= " " and ch != "\x7f")
    # Collapse whitespace
    name = " ".join(name.split())
    # Ensure .pdf extension
    if not name.lower().endswith(".pdf"):
        name += ".pdf"
    # Trim length
    if len(name) > 160:
        stem = Path(name).stem[:140].rstrip()
        name = f"{stem}.pdf"
    return name or "untitled.pdf"

def unique_path(path: Path) -> Path:
    """If `path` exists, append a numeric suffix before the extension."""
    if not path.exists():
        return path
    stem, suffix = path.stem, path.suffix
    k = 2
    while True:
        candidate = path.with_name(f"{stem} ({k}){suffix}")
        if not candidate.exists():
            return candidate
        k += 1

def rename_case_insensitive_safe(src: Path, dst: Path):
    """
    On macOS (often case-insensitive), renaming 'A.pdf' -> 'a.pdf' may be a no-op.
    Use a two-step rename via a temp name when only case differs.
    """
    if src.resolve() == dst.resolve():
        return  # exactly the same path
    if src.parent == dst.parent and src.name.lower() == dst.name.lower() and src.name != dst.name:
        tmp = unique_path(src.with_name(f"__tmp__{int(time.time()*1000)}{src.suffix}"))
        if not DRY_RUN:
            src.rename(tmp)
            tmp.rename(dst)
    else:
        if not DRY_RUN:
            src.rename(dst)

# -------------- Load suggestions ------------
if SUGGESTIONS_XLSX.exists():
    df = pd.read_excel(SUGGESTIONS_XLSX)
elif SUGGESTIONS_CSV.exists():
    df = pd.read_csv(SUGGESTIONS_CSV)
else:
    raise FileNotFoundError("Cannot find deepseek_filename_suggestions.xlsx or .csv under output/")

# Require columns
required = {"file_name", "relative_path", "is_research_paper", "suggested_new_filename"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in suggestions table: {missing}")

# -------------- Process rows ----------------
results = []
total = len(df)
renamed_ok = 0
skipped = 0
failed = 0

for i, row in df.iterrows():
    orig_name = str(row["file_name"])
    rel_path  = str(row["relative_path"])
    is_paper  = bool(row["is_research_paper"])

    # Default result entry
    entry = {
        "index": i,
        "original_file_name": orig_name,
        "relative_path": rel_path,
        "is_research_paper": is_paper,
        "suggested_new_filename": "",
        "final_new_filename": "",
        "status": "",
        "message": ""
    }

    if not is_paper:
        entry["status"] = "skipped"
        entry["message"] = "Not a research paper"
        results.append(entry)
        skipped += 1
        continue

    suggested = str(row.get("suggested_new_filename", "")).strip()
    if not suggested:
        entry["status"] = "skipped"
        entry["message"] = "Empty suggested_new_filename"
        results.append(entry)
        skipped += 1
        continue

    # Resolve source path
    src = BASE_DIR / rel_path
    if not src.exists():
        entry["status"] = "failed"
        entry["message"] = f"Source not found: {src}"
        results.append(entry)
        failed += 1
        continue

    # Sanitize and compute destination path in same folder
    safe_name = sanitize_basename(suggested)
    entry["suggested_new_filename"] = safe_name
    dst = src.with_name(safe_name)

    # Avoid overwriting existing files
    dst_unique = unique_path(dst)
    entry["final_new_filename"] = dst_unique.name

    try:
        if DRY_RUN:
            entry["status"] = "dry-run"
            entry["message"] = f"Would rename to: {dst_unique}"
        else:
            rename_case_insensitive_safe(src, dst_unique)
            entry["status"] = "renamed"
            entry["message"] = f"Renamed to: {dst_unique}"
            renamed_ok += 1
    except Exception as e:
        entry["status"] = "failed"
        entry["message"] = f"Rename error: {e}"
        failed += 1

    results.append(entry)

# -------------- Save report -----------------
df_report = pd.DataFrame(results, columns=[
    "index", "original_file_name", "relative_path", "is_research_paper",
    "suggested_new_filename", "final_new_filename", "status", "message"
])
df_report.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")

print(f"Processed: {total}")
print(f"  Renamed : {renamed_ok}")
print(f"  Skipped : {skipped}")
print(f"  Failed  : {failed}")
print(f"Report saved to: {REPORT_CSV}")


Processed: 420
  Renamed : 332
  Skipped : 88
  Failed  : 0
Report saved to: output/rename_results.csv


In [6]:
# ------------------------------------------------------------
# PDF first-page extraction (notebook-friendly, no main())
# - PyMuPDF (Chinese OK)
# - Suppress MuPDF stderr noise
# - Strong XML-safe sanitizer for Excel (removes C0 & C1 controls)
# - Saves: output/pdf_firstpages.xlsx, .csv, and failures.csv
# ------------------------------------------------------------

from pathlib import Path
import re
import contextlib
import os
import sys
from typing import Dict

import pandas as pd
from tqdm import tqdm

# Prefer the new module name for PyMuPDF
try:
    import pymupdf as fitz  # PyMuPDF >=1.24
except ModuleNotFoundError:
    import fitz  # fallback if older alias is present

# --------- Config ---------
ROOT_DIR = Path("test1")        # folder containing PDFs
OUT_DIR  = Path("output")       # where to save results
XLSX_OUT = OUT_DIR / "pdf_firstpages.xlsx"
CSV_OUT  = OUT_DIR / "pdf_firstpages.csv"
FAIL_OUT = OUT_DIR / "pdf_extraction_failures.csv"
MAX_PAGES = 2                   # extract up to first 2 pages
MAX_CELL = 32000                # Excel cell safe upper bound (32,767)
# --------------------------

# XML 1.0 allowed chars: TAB (0x09), LF (0x0A), CR (0x0D),
# and U+0020..U+D7FF, U+E000..U+FFFD (BMP only; fine for Excel)
XML10_INVALID = re.compile(r'[^\x09\x0A\x0D\x20-\uD7FF\uE000-\uFFFD]')

def xml_safe_text(s: str) -> str:
    """Remove characters disallowed by XML 1.0 (Excel), normalize newlines, strip nulls."""
    if s is None:
        return ""
    # Ensure str, then nuke NULLs and other non-XML characters
    s = str(s)
    s = s.replace("\x00", "")  # fast path for NULLs
    s = XML10_INVALID.sub("", s)
    # Normalize line endings
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    return s

def truncate_for_excel(s: str, max_len: int = MAX_CELL) -> str:
    if not s:
        return s
    return s[:max_len]

@contextlib.contextmanager
def suppress_stderr():
    """Temporarily silence stderr (e.g., noisy MuPDF warnings)."""
    with open(os.devnull, 'w') as devnull:
        old_stderr = sys.stderr
        try:
            sys.stderr = devnull
            yield
        finally:
            sys.stderr = old_stderr

def extract_first_pages_text(pdf_path: Path, max_pages: int = 2) -> Dict[str, str]:
    """
    Return dict with:
      - first_page_text
      - first_two_pages_text
      - ok (bool)
      - reason ('' if ok else category)
    Categories:
      'open_error:*', 'encrypted_no_password', 'auth_error:*',
      'parse_error:*', 'no_text_extracted_maybe_scanned'
    """
    try:
        with suppress_stderr():
            doc = fitz.open(pdf_path)
    except Exception as e:
        return dict(first_page_text="", first_two_pages_text="", ok=False,
                    reason=f"open_error:{type(e).__name__}")

    # Encrypted?
    if getattr(doc, "needs_pass", False):
        try:
            if not doc.authenticate(""):
                doc.close()
                return dict(first_page_text="", first_two_pages_text="", ok=False,
                            reason="encrypted_no_password")
        except Exception as e:
            doc.close()
            return dict(first_page_text="", first_two_pages_text="", ok=False,
                        reason=f"auth_error:{type(e).__name__}")

    try:
        n = min(max_pages, len(doc))
        texts = []
        for i in range(n):
            page = doc[i]
            txt = page.get_text("text") or ""
            texts.append(txt)
        doc.close()

        fp1 = xml_safe_text(texts[0]) if texts else ""
        fp2 = xml_safe_text("\n".join(texts)) if texts else ""
        ok = bool(fp1 or fp2)
        reason = "" if ok else "no_text_extracted_maybe_scanned"
        return dict(first_page_text=fp1, first_two_pages_text=fp2, ok=ok, reason=reason)

    except Exception as e:
        try:
            doc.close()
        except Exception:
            pass
        return dict(first_page_text="", first_two_pages_text="", ok=False,
                    reason=f"parse_error:{type(e).__name__}")

# ---------- Run extraction (cell-safe) ----------
OUT_DIR.mkdir(parents=True, exist_ok=True)
pdf_files = sorted([p for p in ROOT_DIR.rglob("*.pdf")])

print(f"Found {len(pdf_files)} PDF files under '{ROOT_DIR}'. Starting extraction...")

rows = []
fails = []

# Counters for breakdown
counts = {
    "success": 0,
    "open_error": 0,
    "encrypted_no_password": 0,
    "auth_error": 0,
    "parse_error": 0,
    "no_text_extracted_maybe_scanned": 0,
    "other_failure": 0,
}

for pdf in tqdm(pdf_files, desc="Processing PDFs"):
    res = extract_first_pages_text(pdf, MAX_PAGES)

    # Final XML-safe sanitize + length cap just before storing
    fp1 = truncate_for_excel(xml_safe_text(res["first_page_text"]))
    fp2 = truncate_for_excel(xml_safe_text(res["first_two_pages_text"]))

    rows.append({
        "file_name": pdf.name,
        "relative_path": str(pdf.relative_to(ROOT_DIR)),
        "first_page_text": fp1,
        "first_two_pages_text": fp2,
    })

    if res["ok"]:
        counts["success"] += 1
    else:
        reason = res.get("reason", "other_failure")
        if reason.startswith("open_error"):
            counts["open_error"] += 1
        elif reason == "encrypted_no_password":
            counts["encrypted_no_password"] += 1
        elif reason.startswith("auth_error"):
            counts["auth_error"] += 1
        elif reason.startswith("parse_error"):
            counts["parse_error"] += 1
        elif reason == "no_text_extracted_maybe_scanned":
            counts["no_text_extracted_maybe_scanned"] += 1
        else:
            counts["other_failure"] += 1

        fails.append({
            "file_name": pdf.name,
            "relative_path": str(pdf.relative_to(ROOT_DIR)),
            "reason": reason,
        })

# ---------- Save outputs ----------
df = pd.DataFrame(rows, columns=[
    "file_name", "relative_path", "first_page_text", "first_two_pages_text"
])

# Belt & suspenders: XML-safe & truncate again
for col in ("first_page_text", "first_two_pages_text"):
    df[col] = df[col].map(xml_safe_text).map(truncate_for_excel)

# Write files
df.to_excel(XLSX_OUT, index=False, engine="openpyxl")
df.to_csv(CSV_OUT, index=False, encoding="utf-8-sig")

if fails:
    pd.DataFrame(fails, columns=["file_name", "relative_path", "reason"]).to_csv(
        FAIL_OUT, index=False, encoding="utf-8-sig"
    )

# ---------- Summary ----------
total = len(pdf_files)
print("\n========== SUMMARY ==========")
print(f"Total PDFs found:                   {total}")
print(f"Successfully extracted:             {counts['success']}")
failed = total - counts["success"]
print(f"Failed / no text extracted:         {failed}")
if failed:
    print("  Breakdown:")
    print(f"    - open_error:                   {counts['open_error']}")
    print(f"    - encrypted_no_password:        {counts['encrypted_no_password']}")
    print(f"    - auth_error:                   {counts['auth_error']}")
    print(f"    - parse_error:                  {counts['parse_error']}")
    print(f"    - no_text_extracted (scanned):  {counts['no_text_extracted_maybe_scanned']}")
    print(f"    - other_failure:                {counts['other_failure']}")
    print(f"  Failure details saved to:         {FAIL_OUT}")
print("Outputs written:")
print(f" - {XLSX_OUT}")
print(f" - {CSV_OUT}")
print("================================")


Found 420 PDF files under 'test1'. Starting extraction...


Processing PDFs:  82%|████████▏ | 344/420 [00:09<00:03, 22.64it/s]

MuPDF error: format error: object is not a stream

MuPDF error: format error: object is not a stream



Processing PDFs:  91%|█████████▏| 384/420 [00:10<00:00, 41.76it/s]

MuPDF error: format error: object is not a stream

MuPDF error: format error: object is not a stream



Processing PDFs:  98%|█████████▊| 410/420 [00:11<00:00, 34.15it/s]

MuPDF error: format error: object is not a stream

MuPDF error: format error: object is not a stream



Processing PDFs: 100%|██████████| 420/420 [00:11<00:00, 36.12it/s]



========== SUMMARY ==========
Total PDFs found:                   420
Successfully extracted:             420
Failed / no text extracted:         0
Outputs written:
 - output/pdf_firstpages.xlsx
 - output/pdf_firstpages.csv
